In [ ]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv
import pandas as pd,os,json

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
client = Anthropic(api_key=api_key)
model = "claude-haiku-4-5-20251001"


df = pd.read_csv(r"C:\Users\purav\OneDrive\Desktop\Internship\telehealth_patients.csv")

def get_df_info():
    temp_df = df.copy()
    temp_df.fillna("N/A", inplace=True)

    return {
        "columns": temp_df.columns.tolist(),
        "rows": len(temp_df),
        "data_types": temp_df.dtypes.astype(str).to_dict(),
        "top_5_rows": temp_df.head().to_dict(orient="records"),
    }

def run_cohort_analysis(df):
    result={}
    
    for program,group in df.groupby("program_type"):
        
        total_patients=len(group)
        
        active_patients=(
            group["retention_status"].astype(str).str.lower().eq("active").sum()
        )
        
        churned_patients=total_patients-active_patients
        
        retention_rate=round((active_patients/total_patients)*100,2)
        
        result[program]={
            "number_of_patients":total_patients,
            "average_treatment_duration_weeks":round(
                group["treatment_duration_weeks"].mean(),2
            ),
            "active_patients":int(active_patients),
            "churned_patients":int(churned_patients),
            "retention_rate_percent":retention_rate,
        }

    return result

def run_outcome_analysis(df):
    result={}
    
    for program,group in df.groupby("program_type"):
        retention_rate = round(
            (group["retention_status"].astype(str).str.lower().eq("active").sum() / len(group)) * 100, 2
        )
        duration_q1 = round(group["treatment_duration_weeks"].quantile(0.25), 2)
        duration_median = round(group["treatment_duration_weeks"].quantile(0.5), 2)
        duration_q3 = round(group["treatment_duration_weeks"].quantile(0.75), 2)
        duration_mean = round(group["treatment_duration_weeks"].mean(), 2)
        
        early_dropoff = len(group[group["treatment_duration_weeks"] <= duration_q1])
        mid_dropoff = len(group[(group["treatment_duration_weeks"] > duration_q1) & 
                                (group["treatment_duration_weeks"] <= duration_q3)])
        late_dropoff = len(group[group["treatment_duration_weeks"] > duration_q3])
        
        result[program] = {
            "retention_rate_percent": retention_rate,
            "duration_trends": {
                "mean_weeks": duration_mean,
                "median_weeks": duration_median,
                "q1_weeks": duration_q1,
                "q3_weeks": duration_q3,
            },
            "drop_off_points": {
                "early_stage_patients": int(early_dropoff),
                "mid_stage_patients": int(mid_dropoff),
                "late_stage_patients": int(late_dropoff),
            }
        }
    
    return result

tools = [
    {
        "name": "get_df_info",
        "description": "Returns dataframe metadata including columns, row count, data types and first five rows.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "run_cohort_analysis",
        "description": "Segments patients by program (testosterone, weight loss, peptides) and returns group-level stats.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "run_outcome_analysis",
        "description": "Calculates retention rates, treatment duration trends, and drop-off points per program.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]
def add_user_message(messages, message):
    messages.append(
        {
            "role": "user",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def add_assistant_message(messages, message):
    messages.append(
        {
            "role": "assistant",
            "content": message.content if isinstance(message, Message) else message,
        }
    )


def chat(messages, system=None, temperature=0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "tools": tools,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    return client.messages.create(**params)


def text_from_message(message):
    texts = []

    for block in message.content:
        if block.type == "text":
            texts.append(block.text)

    return "\n".join(texts)


def run_tool(tool_name, tool_input):
    if tool_name == "get_df_info":
        return get_df_info()
    elif tool_name =='run_cohort_analysis':
        return run_cohort_analysis(df)
    elif tool_name == 'run_outcome_analysis':
        return run_outcome_analysis(df)

    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(message):
    tool_results = []

    for block in message.content:

        if block.type != "tool_use":
            continue

        try:
            result = run_tool(block.name, block.input)

            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                }
            )

        except Exception as e:
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(e),
                    "is_error": True,
                }
            )

    return tool_results

messages = [
    {
        "role": "user",
        "content": "Analyze the telehealth patient dataset and provide insights on patient retention and treatment outcomes."}
]

response = chat(messages)

if response.stop_reason == "tool_use":

    messages.append(
        {
            "role": "assistant",
            "content": response.content,
        }
    )

    tool_results = run_tools(response)

    messages.append(
        {
            "role": "user",
            "content": tool_results,
        }
    )

    response = chat(messages)

print(text_from_message(response))